In [1]:

!pip install -q numba numpy


In [2]:

!nvidia-smi

import numpy as np
from numba import cuda

print("CUDA available:", cuda.is_available())

if cuda.is_available():
    print("GPU:", cuda.get_current_device().name)


Thu Oct 15 09:40:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
@cuda.jit
def vector_add_1d(a, b, out):
    i = cuda.grid(1)
    if i < out.size:
        out[i] = a[i] + b[i]

n = 1000
a = np.arange(n, dtype=np.float32)
b = np.ones(n, dtype=np.float32) * 2
d_a, d_b = cuda.to_device(a), cuda.to_device(b)
d_out = cuda.device_array(n, dtype=np.float32)
threads = 256
blocks = (n + threads - 1) // threads
vector_add_1d[blocks, threads](d_a, d_b, d_out)
out = d_out.copy_to_host()
np.testing.assert_allclose(out, a + b, atol=1e-5)
print("Lab 1 passed:", out[:5])

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Lab 1 passed: [2. 3. 4. 5. 6.]


In [4]:
@cuda.jit
def vector_scale_grid_stride(x, alpha, out):
    i = cuda.grid(1)
    stride = cuda.gridsize(1)
    while i < x.size:
        out[i] = x[i] * alpha
        i += stride

n = 10000
x = np.arange(n, dtype=np.float32)
alpha = np.float32(3.5)
d_x = cuda.to_device(x)
d_out = cuda.device_array_like(d_x)
vector_scale_grid_stride[64, 256](d_x, alpha, d_out)
out = d_out.copy_to_host()
np.testing.assert_allclose(out, x * alpha, atol=1e-5)
print("Lab 2 passed:", out[:5])

Lab 2 passed: [ 0.   3.5  7.  10.5 14. ]


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 64 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [5]:
@cuda.jit
def matrix_add_2d(A, B, C):
    col, row = cuda.grid(2)
    if row < C.shape[0] and col < C.shape[1]:
        C[row, col] = A[row, col] + B[row, col]

m, n = 37, 53
A = np.random.rand(m, n).astype(np.float32)
B = np.random.rand(m, n).astype(np.float32)
d_A, d_B = cuda.to_device(A), cuda.to_device(B)
d_C = cuda.device_array((m, n), dtype=np.float32)
threads = (16, 16)
blocks = ((n + 15)//16, (m + 15)//16)
matrix_add_2d[blocks, threads](d_A, d_B, d_C)
C = d_C.copy_to_host()
np.testing.assert_allclose(C, A + B, atol=1e-5)
print("Lab 3 passed; output shape:", C.shape)


Lab 3 passed; output shape: (37, 53)


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 12 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [6]:
@cuda.jit
def matrix_transpose_naive(A, B):
    col, row = cuda.grid(2)
    if row < A.shape[0] and col < A.shape[1]:
        B[col, row] = A[row, col]

m, n = 31, 47
A = np.arange(m*n, dtype=np.float32).reshape(m, n)
d_A = cuda.to_device(A)
d_B = cuda.device_array((n, m), dtype=np.float32)
threads = (16, 16)
blocks = ((n + 15)//16, (m + 15)//16)
matrix_transpose_naive[blocks, threads](d_A, d_B)
B = d_B.copy_to_host()
np.testing.assert_allclose(B, A.T)
print("Lab 4 passed; transposed shape:", B.shape)

Lab 4 passed; transposed shape: (47, 31)


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 6 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [7]:
@cuda.jit
def volume_init_3d(vol):
    x, y, z = cuda.grid(3)
    if z < vol.shape[0] and y < vol.shape[1] and x < vol.shape[2]:
        vol[z, y, x] = z * 10000 + y * 100 + x

D, H, W = 9, 11, 13
d_vol = cuda.device_array((D, H, W), dtype=np.int32)
threads = (8, 8, 8)
blocks = ((W+7)//8, (H+7)//8, (D+7)//8)
volume_init_3d[blocks, threads](d_vol)
vol = d_vol.copy_to_host()
z, y, x = np.indices((D, H, W))
expected = z*10000 + y*100 + x
np.testing.assert_array_equal(vol, expected)
print("Lab 5 passed; volume shape:", vol.shape)

Lab 5 passed; volume shape: (9, 11, 13)


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 8 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [8]:
@cuda.jit
def rgb_to_grayscale(img_rgb, img_gray):
    x, y = cuda.grid(2)
    if y < img_rgb.shape[0] and x < img_rgb.shape[1]:
        r = img_rgb[y, x, 0]
        g = img_rgb[y, x, 1]
        b = img_rgb[y, x, 2]
        img_gray[y, x] = 0.299*r + 0.587*g + 0.114*b

h, w = 32, 48
img = np.random.randint(0, 256, size=(h, w, 3)).astype(np.float32)
d_img = cuda.to_device(img)
d_gray = cuda.device_array((h, w), dtype=np.float32)
threads = (16, 16)
blocks = ((w+15)//16, (h+15)//16)
rgb_to_grayscale[blocks, threads](d_img, d_gray)
gray = d_gray.copy_to_host()
expected = 0.299*img[:,:,0] + 0.587*img[:,:,1] + 0.114*img[:,:,2]
np.testing.assert_allclose(gray, expected, atol=1e-4)
print("Lab 6 passed; grayscale shape:", gray.shape)

Lab 6 passed; grayscale shape: (32, 48)


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 6 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [9]:
@cuda.jit
def vector_lerp(a, b, t, out):
    i = cuda.grid(1)
    if i < out.size:
        out[i] = (1.0 - t[i]) * a[i] + t[i] * b[i]

n = 1024
a = np.zeros(n, dtype=np.float32)
b = np.full(n, 10, dtype=np.float32)
t = np.linspace(0, 1, n, dtype=np.float32)
d_a, d_b, d_t = cuda.to_device(a), cuda.to_device(b), cuda.to_device(t)
d_out = cuda.device_array(n, dtype=np.float32)
vector_lerp[(n+255)//256, 256](d_a, d_b, d_t, d_out)
out = d_out.copy_to_host()
np.testing.assert_allclose(out, (1-t)*a + t*b, atol=1e-5)
print("Lab 7 passed:", out[0], out[n//2], out[-1])

Lab 7 passed: 0.0 5.0048876 10.0


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [10]:
@cuda.jit
def vector_threshold(x, thresh, mask):
    i = cuda.grid(1)
    if i < x.size:
        if x[i] >= thresh:
            mask[i] = 1
        else:
            mask[i] = 0

x = np.array([-2, 0, 1, 2.5, 4, 8], dtype=np.float32)
threshold = np.float32(2.0)
d_x = cuda.to_device(x)
d_mask = cuda.device_array(x.size, dtype=np.uint8)
vector_threshold[1, 256](d_x, threshold, d_mask)
mask = d_mask.copy_to_host()
np.testing.assert_array_equal(mask, (x >= threshold).astype(np.uint8))
print("Lab 8 passed:", mask)

Lab 8 passed: [0 0 0 1 1 1]


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [11]:
@cuda.jit
def benchmark_strided_access(in_arr, out_arr, stride):
    i = cuda.grid(1)
    if i < out_arr.size:
        out_arr[i] = in_arr[i * stride]

def time_stride(stride, n=1_000_000, repeats=10):
    source = np.arange(n * stride, dtype=np.float32)
    d_source = cuda.to_device(source)
    d_out = cuda.device_array(n, dtype=np.float32)
    threads = 256
    blocks = (n + threads - 1) // threads
    # Warm-up compilation
    benchmark_strided_access[blocks, threads](d_source, d_out, stride)
    cuda.synchronize()
    start = cuda.event()
    end = cuda.event()
    start.record()
    for _ in range(repeats):
        benchmark_strided_access[blocks, threads](d_source, d_out, stride)
    end.record()
    end.synchronize()
    elapsed_ms = cuda.event_elapsed_time(start, end) / repeats
    result = d_out.copy_to_host()
    np.testing.assert_array_equal(result, source[::stride])
    return elapsed_ms

for stride in (1, 2, 4, 8, 16, 32):
    print(f"stride={stride:2d}: {time_stride(stride):.3f} ms per launch")

stride= 1: 0.061 ms per launch
stride= 2: 0.056 ms per launch
stride= 4: 0.083 ms per launch
stride= 8: 0.148 ms per launch
stride=16: 0.284 ms per launch
stride=32: 0.311 ms per launch


In [15]:

# Lab 10: 1D moving average stencil using shared memory

from numba import cuda, float32
import numpy as np

BLOCK_SIZE = 128
RADIUS = 3

@cuda.jit
def stencil_1d_shared(x, out):
    # Shared memory size must be a compile-time constant
    shared = cuda.shared.array(shape=258, dtype=float32)

    tid = cuda.threadIdx.x
    i = cuda.blockIdx.x * cuda.blockDim.x + tid
    n = x.size

    # Load the central element
    if i < n:
        shared[tid + RADIUS] = x[i]
    else:
        shared[tid + RADIUS] = 0.0

    # Load left halo
    if tid < RADIUS:
        left_i = i - RADIUS
        if left_i >= 0:
            shared[tid] = x[left_i]
        else:
            shared[tid] = x[0]

    # Load right halo
    if tid >= BLOCK_SIZE - RADIUS:
        right_i = i + RADIUS
        halo_pos = tid + 2 * RADIUS
        if right_i < n:
            shared[halo_pos] = x[right_i]
        else:
            shared[halo_pos] = x[n - 1]

    cuda.syncthreads()

    if i < n:
        total = 0.0
        count = 0

        for offset in range(-RADIUS, RADIUS + 1):
            pos = tid + RADIUS + offset

            # Nearest-value boundary handling
            if i + offset < 0:
                value = x[0]
            elif i + offset >= n:
                value = x[n - 1]
            else:
                value = shared[pos]

            total += value
            count += 1

        out[i] = total / count


# Test Lab 10
n = 1000
radius = RADIUS
x = np.arange(n, dtype=np.float32)

d_x = cuda.to_device(x)
d_out = cuda.device_array(n, dtype=np.float32)

blocks = (n + BLOCK_SIZE - 1) // BLOCK_SIZE
stencil_1d_shared[blocks, BLOCK_SIZE](d_x, d_out)

out = d_out.copy_to_host()

# CPU reference for comparison
padded = np.pad(x, (radius, radius), mode="edge")
expected = np.array([
    np.mean(padded[i:i + 2 * radius + 1])
    for i in range(n)
], dtype=np.float32)

np.testing.assert_allclose(out, expected, atol=1e-4)
print("Lab 10 passed: shared-memory stencil is correct!")
print("First 5 values:", out[:5])


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 8 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Lab 10 passed: shared-memory stencil is correct!
First 5 values: [0.85714287 1.4285715  2.142857   3.         4.        ]


In [16]:
!python autograder.py

python3: can't open file '/content/autograder.py': [Errno 2] No such file or directory


In [17]:
!pwd
!ls -la

/content
total 16
drwxr-xr-x 1 root root 4096 Sep 30 13:28 .
drwxr-xr-x 1 root root 4096 Oct 15 09:34 ..
drwxr-xr-x 4 root root 4096 Sep 30 13:28 .config
drwxr-xr-x 1 root root 4096 Sep 30 13:28 sample_data


In [18]:
!find /content -name "autograder.py"

In [19]:
!git clone https://github.com/belkamenbirgee/cuda-lab-02-230103057-

Cloning into 'cuda-lab-02-230103057-'...
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 13 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (13/13), 144.53 KiB | 770.00 KiB/s, done.
Resolving deltas: 100% (1/1), done.


In [20]:
!python autograder.py

python3: can't open file '/content/autograder.py': [Errno 2] No such file or directory


In [21]:
%cd /content
!git clone https://github.com/belkamenbirgee/cuda-lab-02-230103057-


/content
fatal: destination path 'cuda-lab-02-230103057-' already exists and is not an empty directory.


In [22]:
!pwd
!ls -la

/content
total 20
drwxr-xr-x 1 root root 4096 Oct 15 09:54 .
drwxr-xr-x 1 root root 4096 Oct 15 09:34 ..
drwxr-xr-x 4 root root 4096 Sep 30 13:28 .config
drwxr-xr-x 3 root root 4096 Oct 15 09:54 cuda-lab-02-230103057-
drwxr-xr-x 1 root root 4096 Sep 30 13:28 sample_data


In [23]:
!find /content -name "autograder.py"

In [24]:
%cd /content/cuda-lab-02-230103057-
!ls -la

/content/cuda-lab-02-230103057-
total 216
drwxr-xr-x 3 root root   4096 Oct 15 09:54 .
drwxr-xr-x 1 root root   4096 Oct 15 09:54 ..
-rw-r--r-- 1 root root  43255 Oct 15 09:54 CUDA_Lab01_230103057_ipynbCUDA_Lab01_230103057.ipynb
-rw-r--r-- 1 root root 135099 Oct 15 09:54 CUDA_Python_Lab_Manual.docx
drwxr-xr-x 8 root root   4096 Oct 15 09:54 .git
-rw-r--r-- 1 root root   2351 Oct 15 09:54 README.md
-rw-r--r-- 1 root root   5408 Oct 15 09:54 task1_divergence.py
-rw-r--r-- 1 root root   2678 Oct 15 09:54 task2_stencil_1d.py
-rw-r--r-- 1 root root   2287 Oct 15 09:54 task3_grid_stride.py
-rw-r--r-- 1 root root   3871 Oct 15 09:54 task4_sobel_2d.py
-rw-r--r-- 1 root root   3873 Oct 15 09:54 verify_submission.py


In [25]:
%cd /content/cuda-lab-02-230103057-
!ls -R

/content/cuda-lab-02-230103057-
.:
CUDA_Lab01_230103057_ipynbCUDA_Lab01_230103057.ipynb  task2_stencil_1d.py
CUDA_Python_Lab_Manual.docx			      task3_grid_stride.py
README.md					      task4_sobel_2d.py
task1_divergence.py				      verify_submission.py


In [26]:
!python generate_submission_token.py "230103057" "Praliyeva Nuray"

python3: can't open file '/content/cuda-lab-02-230103057-/generate_submission_token.py': [Errno 2] No such file or directory


In [27]:
%cd /content/cuda-lab-02-230103057-
!git status
!git remote -v

/content/cuda-lab-02-230103057-
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
origin	https://github.com/belkamenbirgee/cuda-lab-02-230103057- (fetch)
origin	https://github.com/belkamenbirgee/cuda-lab-02-230103057- (push)


In [28]:
%cd /content/cuda-lab-02-230103057-
!ls -la

/content/cuda-lab-02-230103057-
total 216
drwxr-xr-x 3 root root   4096 Oct 15 09:54 .
drwxr-xr-x 1 root root   4096 Oct 15 09:54 ..
-rw-r--r-- 1 root root  43255 Oct 15 09:54 CUDA_Lab01_230103057_ipynbCUDA_Lab01_230103057.ipynb
-rw-r--r-- 1 root root 135099 Oct 15 09:54 CUDA_Python_Lab_Manual.docx
drwxr-xr-x 8 root root   4096 Oct 15 10:04 .git
-rw-r--r-- 1 root root   2351 Oct 15 09:54 README.md
-rw-r--r-- 1 root root   5408 Oct 15 09:54 task1_divergence.py
-rw-r--r-- 1 root root   2678 Oct 15 09:54 task2_stencil_1d.py
-rw-r--r-- 1 root root   2287 Oct 15 09:54 task3_grid_stride.py
-rw-r--r-- 1 root root   3871 Oct 15 09:54 task4_sobel_2d.py
-rw-r--r-- 1 root root   3873 Oct 15 09:54 verify_submission.py


In [ ]:
from google.colab import files
uploaded = files.upload()